# 01 — Dữ liệu, chia tập và EDA
Người 1. Dùng CPU; chỉ phân tích đặc trưng trên Train.


## Mở project
**Colab:** đặt đầy đủ thư mục project tại `MyDrive/student-dropout-prediction`,
gồm notebook, `common/`, `requirements.txt` và `data/raw/data.csv`.
Nếu thư mục có tên khác, sửa `PROJECT_ROOT` ở cell dưới. Mở mỗi notebook rồi chạy từ trên xuống.
**VS Code/Jupyter:** mở notebook từ thư mục `notebooks/`; cell tự tìm thư mục gốc.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/student-dropout-prediction')
else:
    here = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in [here, *here.parents]
                         if (p / 'common' / 'data_io.ipynb').is_file()), here)

if not (PROJECT_ROOT / 'common' / 'data_io.ipynb').is_file():
    raise FileNotFoundError('Sửa PROJECT_ROOT tới thư mục chứa common/ và requirements.txt.')
os.chdir(PROJECT_ROOT)
print('Project:', PROJECT_ROOT)


## Thư viện
Colab không cần `venv`, nhưng vẫn cần đúng thư viện. Cell này đọc `requirements.txt`.
Nếu Colab yêu cầu khởi động lại runtime sau khi cài, thực hiện rồi chạy lại từ đầu.


In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])


In [ ]:
import json
import pandas as pd
from IPython.display import display
%run common/data_io.ipynb
%run common/splits.ipynb

DATA_PATH = PROJECT_ROOT / 'data/raw/data.csv'
SPLIT_PATH = PROJECT_ROOT / 'data/splits/splits.json'
ARTIFACTS = PROJECT_ROOT / 'artifacts'
TABLES = PROJECT_ROOT / 'reports/tables'
FIGURES = PROJECT_ROOT / 'reports/figures'
for folder in [SPLIT_PATH.parent, ARTIFACTS, TABLES, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Đặt CSV gốc tại {DATA_PATH}')


## Kiểm tra đầu vào


In [ ]:
%run common/preprocessing.ipynb
df = load_data(DATA_PATH)
print('Kích thước:', df.shape)
print('Dòng trùng:', int(df.duplicated().sum()))
print('Tổng giá trị thiếu:', int(df.isna().sum().sum()))
print('Nhãn hợp lệ:', LABELS)


## Bộ chia chung
Seed 42. Tỉ lệ 70/15/15 dưới đây là ví dụ, cần nhóm thống nhất trước khi chạy chính thức.
Chỉ số dòng bắt đầu từ 0, không tính dòng tiêu đề CSV. File chia hiện có được dùng lại;
code báo lỗi nếu CSV hoặc cấu hình khác để tránh vô tình tạo lại phép chia.


In [ ]:
SEED = 42
TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15
if SPLIT_PATH.exists():
    split = load_splits(SPLIT_PATH, DATA_PATH, df)
    if (split['seed'], split['test_size'], split['validation_size']) != (SEED, TEST_SIZE, VALIDATION_SIZE):
        raise ValueError('Cấu hình khác bộ chia đã lưu. Dùng cấu hình đã chốt của nhóm.')
else:
    split = create_splits(df, DATA_PATH, SPLIT_PATH, SEED, TEST_SIZE, VALIDATION_SIZE)
print({part: len(split[part]) for part in ['train', 'validation', 'test']})
print('Số fold chung:', len(split['folds']))


## EDA trên Train và danh sách cột


In [ ]:
train = df.iloc[split['train']].copy()
numeric, categorical = feature_groups(train.drop(columns=TARGET))
binary = [c for c in categorical if c in BINARY]
overview = {
    'full_shape': list(df.shape), 'full_duplicate_rows': int(df.duplicated().sum()),
    'train_shape': list(train.shape), 'train_duplicate_rows': int(train.duplicated().sum()),
    'numeric_columns': numeric,
    'categorical_columns': [c for c in categorical if c not in binary],
    'binary_columns': binary,
}
eda_dir = TABLES / 'eda_train'
eda_dir.mkdir(exist_ok=True)
save_json(eda_dir / 'overview.json', overview)
train.isna().sum().rename('missing_count').to_csv(eda_dir / 'missing.csv')
train[TARGET].value_counts().reindex(LABELS, fill_value=0).rename('count').to_csv(eda_dir / 'target_counts.csv')
train[TARGET].value_counts(normalize=True).reindex(LABELS, fill_value=0).rename('proportion').to_csv(eda_dir / 'class_proportions.csv')
summary = train.describe(include='all').transpose()
summary.to_csv(eda_dir / 'summary.csv')
display(pd.DataFrame([(kind, cols) for kind, cols in overview.items() if kind.endswith('_columns')], columns=['group', 'columns']))
display(summary)


In [ ]:
import matplotlib.pyplot as plt
figure_dir = FIGURES / 'eda_train'
figure_dir.mkdir(exist_ok=True)
ax = train[TARGET].value_counts().reindex(LABELS, fill_value=0).plot.bar(color='#3975b4')
ax.set(title='Phân bố nhãn trên Train', xlabel='Target', ylabel='Số mẫu')
plt.tight_layout()
plt.savefig(figure_dir / 'target_distribution.png', dpi=160)
plt.show()
if numeric:
    axes = train[numeric[:6]].hist(figsize=(12, 7), bins=20)
    plt.tight_layout()
    plt.savefig(figure_dir / 'numeric_histograms.png', dpi=160)
    plt.show()


## Ghi nhận xét sau khi chạy
Bổ sung nhận xét từ bảng và biểu đồ: mất cân bằng nhãn, giá trị thiếu, dòng trùng,
đặc trưng cần xử lý. Nếu thay đổi CSV hoặc xử lý dòng trùng làm đổi thứ tự/số dòng,
nhóm phải tạo bộ chia mới trước khi train. Không suy ra kết quả từ notebook chưa chạy.
Bàn giao `data/splits/splits.json`, đúng CSV gốc và các bảng/biểu đồ EDA.
